# Propiedades de grafos

Un grafo $G = (V, E)$ es un conjunto de vértices y otro de aristas que los unen. Se guarda como lista de aristas, como lista de adyacencia (cada vértice con sus vecinos, $O(V + E)$ de memoria) o como matriz de adyacencia ($O(V^2)$ de memoria, pero dice en $O(1)$ si dos vértices son vecinos). Este notebook convierte entre las tres, construye las familias clásicas (completo, ciclo, camino, rueda y bipartito completo) y calcula grados, regularidad, complemento, conexión y subgrafos inducidos; casi todo cuesta $O(V + E)$ con la lista de adyacencia. El resultado que más se repite es el lema del apretón de manos: $\sum_v \deg(v) = 2|E|$. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python); `networkx` se usa solo para contrastar los resultados.

## 1. Tres representaciones

Representar una red de ferris inventada entre las siete islas como lista de aristas, lista de adyacencia (un `dict` de conjuntos) y matriz de adyacencia, y escribir las conversiones entre ellas. Como el grafo no es dirigido, la arista `("GC", "TF")` es la misma que `("TF", "GC")`: normalizarlas ordenando sus extremos. Comprobar que al convertir ida y vuelta se recupera el mismo grafo y que la matriz es simétrica.

In [1]:
import itertools
import random
from collections import Counter, deque
from math import comb, inf

import networkx as nx

ISLAS = ["LZ", "FV", "GC", "TF", "LG", "LP", "EH"]
FERRIS = [("LZ", "FV"), ("FV", "GC"), ("GC", "TF"), ("TF", "LG"), ("TF", "LP"),
          ("LG", "EH"), ("LG", "LP"), ("GC", "LZ"), ("TF", "EH")]


def normalizar(aristas):
    return {tuple(sorted(a)) for a in aristas}


def a_adyacencia(vertices, aristas, dirigido=False):
    ady = {v: set() for v in vertices}
    for u, v in aristas:
        ady[u].add(v)
        if not dirigido:
            ady[v].add(u)
    return ady


def de_adyacencia(ady, dirigido=False):
    aristas = {(u, v) for u, vecinos in ady.items() for v in vecinos}
    return aristas if dirigido else normalizar(aristas)


def a_matriz(vertices, ady):
    indice = {v: i for i, v in enumerate(vertices)}
    m = [[0] * len(vertices) for _ in vertices]
    for u, vecinos in ady.items():
        for v in vecinos:
            m[indice[u]][indice[v]] = 1
    return m


def de_matriz(vertices, m):
    return {u: {vertices[j] for j, x in enumerate(fila) if x} for u, fila in zip(vertices, m)}


def mostrar_matriz(vertices, m, celda=str):
    print("    " + "".join(f"{v:>5}" for v in vertices))
    for v, fila in zip(vertices, m):
        print(f"{v:>4}" + "".join(f"{celda(x):>5}" for x in fila))


ady = a_adyacencia(ISLAS, FERRIS)
for isla in ISLAS:
    print(f"{isla}: {', '.join(sorted(ady[isla]))}")
m = a_matriz(ISLAS, ady)
print()
mostrar_matriz(ISLAS, m)

assert de_adyacencia(ady) == normalizar(FERRIS)
assert de_matriz(ISLAS, m) == ady
assert all(m[i][j] == m[j][i] for i in range(len(ISLAS)) for j in range(len(ISLAS)))

LZ: FV, GC
FV: GC, LZ
GC: FV, LZ, TF
TF: EH, GC, LG, LP
LG: EH, LP, TF
LP: LG, TF
EH: LG, TF

       LZ   FV   GC   TF   LG   LP   EH
  LZ    0    1    1    0    0    0    0
  FV    1    0    1    0    0    0    0
  GC    1    1    0    1    0    0    0
  TF    0    0    1    0    1    1    1
  LG    0    0    0    1    0    1    1
  LP    0    0    0    1    1    0    0
  EH    0    0    0    1    1    0    0


## 2. Grafos dirigidos y ponderados

Unas líneas de guagua inventadas entre pueblos de Gran Canaria forman un grafo dirigido y ponderado por los minutos de cada trayecto: no todas tienen vuelta y la vuelta puede tardar distinto. Representarlo como lista de aristas `(origen, destino, minutos)` y como diccionario `{origen: {destino: minutos}}`, mostrar su matriz de pesos (con `inf` donde no hay trayecto directo) y obtener el grafo no dirigido subyacente, que olvida los sentidos. Comprobar con grafos dirigidos aleatorios que un grafo dirigido tiene cada arista acompañada de su contraria si y solo si su matriz es simétrica.

In [2]:
LINEAS = [("LPA", "TEL", 25), ("TEL", "LPA", 30), ("TEL", "AGÜ", 20), ("AGÜ", "TEL", 20),
          ("LPA", "ARU", 30), ("ARU", "GÁL", 25), ("GÁL", "AGA", 15), ("AGA", "GÁL", 15), ("GÁL", "LPA", 50)]
PUEBLOS = sorted({p for u, v, _ in LINEAS for p in (u, v)})


def a_ponderado(aristas):
    g = {}
    for u, v, w in aristas:
        g.setdefault(u, {})[v] = w
        g.setdefault(v, {})
    return g


def matriz_pesos(vertices, g):
    return [[0 if u == v else g[u].get(v, inf) for v in vertices] for u in vertices]


def subyacente(g):
    return normalizar((u, v) for u in g for v in g[u])


lineas = a_ponderado(LINEAS)
mostrar_matriz(PUEBLOS, matriz_pesos(PUEBLOS, lineas), celda=lambda x: "·" if x == inf else x)
print("\nSubyacente:", sorted(subyacente(lineas)))

assert sorted((u, v, w) for u in lineas for v, w in lineas[u].items()) == sorted(LINEAS)
D = nx.DiGraph()
D.add_weighted_edges_from(LINEAS)
assert subyacente(lineas) == normalizar(D.to_undirected().edges())

rng = random.Random(1)
for _ in range(500):
    n = rng.randint(1, 6)
    arcos = {(u, v) for u in range(n) for v in range(n) if u != v and rng.random() < 0.4}
    if rng.random() < 0.5:
        arcos |= {(v, u) for u, v in arcos}
    m = a_matriz(range(n), a_adyacencia(range(n), arcos, dirigido=True))
    simetrica = all(m[i][j] == m[j][i] for i in range(n) for j in range(n))
    assert simetrica == all((v, u) in arcos for u, v in arcos)

      AGA  AGÜ  ARU  GÁL  LPA  TEL
 AGA    0    ·    ·   15    ·    ·
 AGÜ    ·    0    ·    ·    ·   20
 ARU    ·    ·    0   25    ·    ·
 GÁL   15    ·    ·    0   50    ·
 LPA    ·    ·   30    ·    0   25
 TEL    ·   20    ·    ·   30    0

Subyacente: [('AGA', 'GÁL'), ('AGÜ', 'TEL'), ('ARU', 'GÁL'), ('ARU', 'LPA'), ('GÁL', 'LPA'), ('LPA', 'TEL')]


## 3. Familias de grafos

Escribir funciones que devuelvan los vértices $0, 1, \dots$ y las aristas de:

- el grafo completo $K_n$, con todas las aristas posibles: $\binom{n}{2}$;
- el ciclo $C_n$ ($n \ge 3$): $n$ aristas;
- el camino $P_n$: $n - 1$ aristas;
- la rueda $W_n$ ($n \ge 4$): un centro $0$ unido a un ciclo $1, \dots, n-1$, con $2(n - 1)$ aristas;
- el bipartito completo $K_{m,n}$, que une cada uno de los $m$ primeros vértices con cada uno de los $n$ siguientes: $m \cdot n$ aristas.

Comprobar el número de aristas y que coinciden arista a arista con los generadores de `networkx`, que usan el mismo etiquetado.

In [3]:
def completo(n):
    return list(range(n)), {(i, j) for i in range(n) for j in range(i + 1, n)}


def ciclo(n):
    return list(range(n)), normalizar((i, (i + 1) % n) for i in range(n))


def camino(n):
    return list(range(n)), {(i, i + 1) for i in range(n - 1)}


def rueda(n):
    _, borde = ciclo(n - 1)
    return list(range(n)), {(0, i) for i in range(1, n)} | {(a + 1, b + 1) for a, b in borde}


def bipartito_completo(m, n):
    return list(range(m + n)), {(i, j) for i in range(m) for j in range(m, m + n)}


familias = [("K(6)", completo(6), comb(6, 2)), ("C(6)", ciclo(6), 6), ("P(6)", camino(6), 5),
            ("W(6)", rueda(6), 10), ("K(3,4)", bipartito_completo(3, 4), 12)]
print(f"{'grafo':<8}{'V':>3}{'E':>4}")
for nombre, (v, a), esperado in familias:
    print(f"{nombre:<8}{len(v):>3}{len(a):>4}")
    assert len(a) == esperado

for n in range(4, 10):
    assert completo(n)[1] == normalizar(nx.complete_graph(n).edges())
    assert ciclo(n)[1] == normalizar(nx.cycle_graph(n).edges())
    assert camino(n)[1] == normalizar(nx.path_graph(n).edges())
    assert rueda(n)[1] == normalizar(nx.wheel_graph(n).edges())
for m, n in itertools.product(range(1, 5), repeat=2):
    assert bipartito_completo(m, n)[1] == normalizar(nx.complete_bipartite_graph(m, n).edges())

grafo     V   E
K(6)      6  15
C(6)      6   6
P(6)      6   5
W(6)      6  10
K(3,4)    7  12


## 4. Incidentes, contiguos y grado

En un grafo no dirigido, las aristas incidentes en $v$ son las que lo tienen como extremo, dos vértices son contiguos (adyacentes) si hay una arista entre ellos y el grado de $v$ es su número de aristas incidentes. Escribir `incidentes`, `contiguos` y `grado`, mostrar las islas de la red de ferris ordenadas por grado y comprobarlo contra `networkx`.

In [4]:
def incidentes(aristas, v):
    return {a for a in normalizar(aristas) if v in a}


def contiguos(ady, u, v):
    return v in ady[u]


def grado(ady, v):
    return len(ady[v])


for isla in sorted(ISLAS, key=lambda i: (-grado(ady, i), i)):
    print(f"{isla}: grado {grado(ady, isla)}  aristas {sorted(incidentes(FERRIS, isla))}")
print("\n¿Hay ferri directo LZ-TF?", contiguos(ady, "LZ", "TF"), " ¿Y GC-TF?", contiguos(ady, "GC", "TF"))

G = nx.Graph(FERRIS)
assert all(grado(ady, v) == len(incidentes(FERRIS, v)) == G.degree[v] for v in ISLAS)
assert all(contiguos(ady, u, v) == G.has_edge(u, v) for u in ISLAS for v in ISLAS)

TF: grado 4  aristas [('EH', 'TF'), ('GC', 'TF'), ('LG', 'TF'), ('LP', 'TF')]
GC: grado 3  aristas [('FV', 'GC'), ('GC', 'LZ'), ('GC', 'TF')]
LG: grado 3  aristas [('EH', 'LG'), ('LG', 'LP'), ('LG', 'TF')]
EH: grado 2  aristas [('EH', 'LG'), ('EH', 'TF')]
FV: grado 2  aristas [('FV', 'GC'), ('FV', 'LZ')]
LP: grado 2  aristas [('LG', 'LP'), ('LP', 'TF')]
LZ: grado 2  aristas [('FV', 'LZ'), ('GC', 'LZ')]

¿Hay ferri directo LZ-TF? False  ¿Y GC-TF? True


## 5. Grados de entrada y de salida

En un grafo dirigido, los sucesores de $v$ son los vértices a los que sale una arista desde $v$ y los predecesores, aquellos desde los que llega una; sus cantidades son el grado de salida y el de entrada. Calcularlos para las líneas de guagua y señalar los pueblos cuyos sucesores y predecesores no coinciden (no se puede volver por la misma línea). Comprobar con grafos dirigidos aleatorios que la suma de los grados de entrada y la de los de salida valen ambas $|E|$, contrastando los grados con `networkx`.

In [5]:
def sucesores(g, v):
    return set(g[v])


def predecesores(g, v):
    return {u for u in g if v in g[u]}


print(f"{'pueblo':<7}{'sale':>5}{'entra':>6}  sucesores        predecesores")
for p in PUEBLOS:
    suc, pre = sucesores(lineas, p), predecesores(lineas, p)
    marca = " ←" if suc != pre else ""
    print(f"{p:<7}{len(suc):>5}{len(pre):>6}  {', '.join(sorted(suc)):<16} {', '.join(sorted(pre))}{marca}")

rng = random.Random(2)
for _ in range(300):
    n = rng.randint(1, 8)
    arcos = [(u, v) for u in range(n) for v in range(n) if u != v and rng.random() < 0.3]
    g = a_ponderado((u, v, 1) for u, v in arcos)
    g |= {v: {} for v in range(n) if v not in g}
    D = nx.DiGraph()
    D.add_nodes_from(range(n))
    D.add_edges_from(arcos)
    assert sum(len(sucesores(g, v)) for v in g) == sum(len(predecesores(g, v)) for v in g) == len(arcos)
    assert all(len(sucesores(g, v)) == D.out_degree[v] and len(predecesores(g, v)) == D.in_degree[v] for v in g)

pueblo  sale entra  sucesores        predecesores
AGA        1     1  GÁL              GÁL
AGÜ        1     1  TEL              TEL
ARU        1     1  GÁL              LPA ←
GÁL        2     2  AGA, LPA         AGA, ARU ←
LPA        2     2  ARU, TEL         GÁL, TEL ←
TEL        2     2  AGÜ, LPA         AGÜ, LPA


## 6. Lazos y aristas múltiples

Un multigrafo admite lazos (aristas de un vértice a sí mismo) y varias aristas entre los mismos dos vértices. Con los senderos entre pueblos de la cumbre de Gran Canaria (hay una ruta circular que sale y vuelve a Ayacata y varios senderos distintos entre Cruz de Tejeda y Tejeda), escribir `lazos`, `aristas_multiples` (con su número de copias), `es_simple` y `simplificar`, que quite lazos y repeticiones. El grado cuenta las aristas incidentes y cada lazo suma 2, porque toca dos veces el vértice; comprobarlo contra `nx.MultiGraph`.

In [6]:
SENDEROS = [("Tejeda", "Artenara"), ("Artenara", "Tejeda"), ("Tejeda", "Ayacata"), ("Ayacata", "Ayacata"),
            ("Ayacata", "Cruz de Tejeda"), ("Cruz de Tejeda", "Tejeda"), ("Tejeda", "Cruz de Tejeda"),
            ("Cruz de Tejeda", "Tejeda")]


def lazos(aristas):
    return [a for a in aristas if a[0] == a[1]]


def aristas_multiples(aristas):
    cuenta = Counter(tuple(sorted(a)) for a in aristas)
    return {a: k for a, k in cuenta.items() if k > 1}


def es_simple(aristas):
    return not lazos(aristas) and not aristas_multiples(aristas)


def simplificar(aristas):
    return {tuple(sorted(a)) for a in aristas if a[0] != a[1]}


def grados_multigrafo(aristas):
    grados = Counter()
    for u, v in aristas:
        grados[u] += 1
        grados[v] += 1
    return grados


print("Lazos:", lazos(SENDEROS))
print("Aristas múltiples:", aristas_multiples(SENDEROS))
print("Simplificado:", sorted(simplificar(SENDEROS)), "¿simple?", es_simple(simplificar(SENDEROS)))
for pueblo, g in sorted(grados_multigrafo(SENDEROS).items()):
    print(f"  {pueblo:<15} grado {g}")

M = nx.MultiGraph(SENDEROS)
assert dict(M.degree) == grados_multigrafo(SENDEROS)
assert not es_simple(SENDEROS) and es_simple(simplificar(SENDEROS))
assert simplificar(SENDEROS) == normalizar(nx.Graph(M).edges()) - {("Ayacata", "Ayacata")}

Lazos: [('Ayacata', 'Ayacata')]
Aristas múltiples: {('Artenara', 'Tejeda'): 2, ('Cruz de Tejeda', 'Tejeda'): 3}
Simplificado: [('Artenara', 'Tejeda'), ('Ayacata', 'Cruz de Tejeda'), ('Ayacata', 'Tejeda'), ('Cruz de Tejeda', 'Tejeda')] ¿simple? True
  Artenara        grado 2
  Ayacata         grado 4
  Cruz de Tejeda  grado 4
  Tejeda          grado 6


## 7. Lema del apretón de manos

En cualquier grafo, también con lazos y aristas múltiples, $\sum_v \deg(v) = 2|E|$, porque cada arista suma 1 a cada uno de sus extremos. Como la suma es par, el número de vértices de grado impar también lo es. Comprobarlo con 1000 grafos y multigrafos aleatorios, mostrando los primeros, y responder: ¿pueden 7 personas saludarse de modo que cada una dé la mano exactamente a otras 3?

In [7]:
def grafo_aleatorio(n, p, rng):
    return {(i, j) for i in range(n) for j in range(i + 1, n) if rng.random() < p}


def multigrafo_aleatorio(n, m, rng):
    return [(rng.randrange(n), rng.randrange(n)) for _ in range(m)]


rng = random.Random(7)
print(f"{'tipo':<11}{'V':>3}{'E':>4}{'Σ grados':>10}{'impares':>9}")
for k in range(1000):
    n = rng.randint(1, 12)
    aristas = grafo_aleatorio(n, rng.random(), rng) if k % 2 else multigrafo_aleatorio(n, rng.randint(0, 20), rng)
    grados = grados_multigrafo(aristas)
    impares = sum(1 for v in range(n) if grados[v] % 2)
    assert sum(grados.values()) == 2 * len(aristas)
    assert impares % 2 == 0
    if k < 6:
        print(f"{'simple' if k % 2 else 'multigrafo':<11}{n:>3}{len(aristas):>4}{sum(grados.values()):>10}{impares:>9}")

print("\n¿7 personas con 3 saludos cada una?", "sí" if 7 * 3 % 2 == 0 else "no: 7 · 3 = 21 es impar y tendría que ser 2|E|")

tipo         V   E  Σ grados  impares
multigrafo   6   4         8        4
simple       1   0         0        0
multigrafo   4   1         2        2
simple       7   1         2        2
multigrafo   9   2         4        4
simple       7  17        34        4

¿7 personas con 3 saludos cada una? no: 7 · 3 = 21 es impar y tendría que ser 2|E|


## 8. Grafos regulares

Un grafo es regular si todos sus vértices tienen el mismo grado, y $k$-regular si ese grado es $k$. Comprobar qué familias del ejercicio 3 son regulares (ojo con la rueda de 4 vértices, que es $K_4$). Por el apretón de manos, un grafo $k$-regular con $n$ vértices exige $n \cdot k$ par (y $k < n$). Escribir `regular(n, k)`, que en ese caso construya uno colocando los vértices en círculo y uniendo cada uno con los $\lfloor k/2 \rfloor$ siguientes (y con el opuesto si $k$ es impar), y que lance `ValueError` si no existe. Comprobarlo para todo $n \le 12$.

In [8]:
def regularidad(vertices, aristas):
    ady = a_adyacencia(vertices, aristas)
    grados = {len(ady[v]) for v in vertices}
    return grados.pop() if len(grados) == 1 else None


def regular(n, k):
    if not 0 <= k < n or n * k % 2:
        raise ValueError(f"no existe un grafo {k}-regular con {n} vértices")
    aristas = {tuple(sorted((i, (i + d) % n))) for i in range(n) for d in range(1, k // 2 + 1)}
    if k % 2:
        aristas |= {(i, i + n // 2) for i in range(n // 2)}
    return list(range(n)), aristas


print(f"{'n':>2} {'K(n)':>5} {'C(n)':>5} {'P(n)':>5} {'W(n)':>5} {'K(n,n)':>7}")
for n in range(4, 8):
    fila = [regularidad(*f) for f in (completo(n), ciclo(n), camino(n), rueda(n), bipartito_completo(n, n))]
    print(f"{n:>2} " + " ".join(f"{'-' if r is None else r:>{w}}" for r, w in zip(fila, (5, 5, 5, 5, 7))))
    assert fila == [n - 1, 2, None, 3 if n == 4 else None, n]

for n in range(1, 13):
    for k in range(n):
        if n * k % 2:
            try:
                regular(n, k)
            except ValueError:
                continue
            raise AssertionError((n, k))
        v, a = regular(n, k)
        assert regularidad(v, a) == k and len(a) == n * k // 2
try:
    regular(7, 3)
except ValueError as e:
    print("\nregular(7, 3):", e)
print("Un grafo 3-regular con 8 vértices:", sorted(regular(8, 3)[1]))

 n  K(n)  C(n)  P(n)  W(n)  K(n,n)
 4     3     2     -     3       4
 5     4     2     -     -       5
 6     5     2     -     -       6
 7     6     2     -     -       7

regular(7, 3): no existe un grafo 3-regular con 7 vértices
Un grafo 3-regular con 8 vértices: [(0, 1), (0, 4), (0, 7), (1, 2), (1, 5), (2, 3), (2, 6), (3, 4), (3, 7), (4, 5), (5, 6), (6, 7)]


## 9. Secuencia de grados e isomorfismo

La secuencia de grados es la lista de grados ordenada de mayor a menor. Dos grafos isomorfos (iguales salvo el nombre de los vértices) tienen la misma secuencia, pero no al revés: $C_6$ y dos triángulos separados tienen ambos $[2, 2, 2, 2, 2, 2]$. Escribir `isomorfos`, que descarte primero por número de vértices, de aristas y por la secuencia, y si no, pruebe todas las biyecciones entre los vértices. Comprobar que $C_6$ y los dos triángulos no son isomorfos y contrastarlo con `nx.is_isomorphic` en grafos aleatorios.

In [9]:
def secuencia_grados(vertices, aristas):
    ady = a_adyacencia(vertices, aristas)
    return sorted((len(ady[v]) for v in vertices), reverse=True)


def isomorfos(v1, a1, v2, a2):
    if len(v1) != len(v2) or len(a1) != len(a2) or secuencia_grados(v1, a1) != secuencia_grados(v2, a2):
        return False
    a2 = normalizar(a2)
    return any(normalizar((f[u], f[v]) for u, v in a1) == a2
               for f in (dict(zip(v1, p)) for p in itertools.permutations(v2)))


DOS_TRIANGULOS = (list(range(6)), {(0, 1), (1, 2), (0, 2), (3, 4), (4, 5), (3, 5)})
print("C(6):          ", secuencia_grados(*ciclo(6)))
print("Dos triángulos:", secuencia_grados(*DOS_TRIANGULOS))
print("¿Isomorfos?", isomorfos(*ciclo(6), *DOS_TRIANGULOS))
print("¿C(6) es isomorfo a sí mismo con otros nombres?",
      isomorfos(*ciclo(6), list("abcdef"), normalizar(zip("abcdef", "bcdefa"))))
assert not isomorfos(*ciclo(6), *DOS_TRIANGULOS)


def a_nx(vertices, aristas):
    G = nx.Graph()
    G.add_nodes_from(vertices)
    G.add_edges_from(aristas)
    return G


rng = random.Random(9)
iguales = 0
for _ in range(300):
    n = rng.randint(1, 6)
    v, a1 = list(range(n)), grafo_aleatorio(n, 0.5, rng)
    if rng.random() < 0.5:
        p = rng.sample(v, n)
        a2 = normalizar((p[x], p[y]) for x, y in a1)
    else:
        a2 = grafo_aleatorio(n, 0.5, rng)
    resultado = isomorfos(v, a1, v, a2)
    iguales += resultado
    assert resultado == nx.is_isomorphic(a_nx(v, a1), a_nx(v, a2))
print(f"{iguales} de 300 parejas aleatorias eran isomorfas")

C(6):           [2, 2, 2, 2, 2, 2]
Dos triángulos: [2, 2, 2, 2, 2, 2]
¿Isomorfos? False
¿C(6) es isomorfo a sí mismo con otros nombres? True
200 de 300 parejas aleatorias eran isomorfas


## 10. Secuencias gráficas: Havel-Hakimi

Una secuencia de enteros es gráfica si existe un grafo simple con esos grados. El teorema de Havel-Hakimi lo decide: se quita el mayor valor $d$ y se resta 1 a los $d$ siguientes; la secuencia es gráfica si y solo si la que queda lo es. Escribir `havel_hakimi`, que devuelva la respuesta y los pasos, y `realizar`, que construya un grafo con esa secuencia uniendo en cada paso el vértice de mayor grado pendiente con los $d$ siguientes. Comprobarlo contra la fuerza bruta (las secuencias de todos los grafos de hasta 5 vértices) y contra `nx.is_graphical`.

In [10]:
def havel_hakimi(secuencia):
    s = sorted(secuencia, reverse=True)
    pasos = [s]
    while s and s[0] > 0:
        d, resto = s[0], s[1:]
        if d > len(resto) or resto[d - 1] == 0:
            return False, pasos
        s = sorted([x - 1 for x in resto[:d]] + resto[d:], reverse=True)
        pasos.append(s)
    return all(x == 0 for x in s), pasos


def realizar(secuencia):
    pendientes = [[d, v] for v, d in enumerate(secuencia)]
    aristas = set()
    while pendientes:
        pendientes.sort(reverse=True)
        d, v = pendientes.pop(0)
        if d < 0 or d > len(pendientes) or (d and pendientes[d - 1][0] == 0):
            return None
        for par in pendientes[:d]:
            par[0] -= 1
            aristas.add(tuple(sorted((v, par[1]))))
    return aristas


for s in ([3, 3, 2, 2, 2, 1, 1], [4, 4, 4, 1, 1], [5, 3, 2, 2, 1, 1], [3, 2, 2]):
    ok, pasos = havel_hakimi(s)
    print(f"{s}: {'gráfica' if ok else 'no gráfica'}")
    print("   " + " → ".join(map(str, pasos)))
    aristas = realizar(s)
    assert (aristas is not None) == ok == nx.is_graphical(s)
    if ok:
        print("   aristas:", sorted(aristas))
        assert secuencia_grados(range(len(s)), aristas) == sorted(s, reverse=True)

print(f"\n{'n':>2} {'grafos':>7} {'secuencias gráficas':>20}")
for n in range(1, 6):
    pares = list(itertools.combinations(range(n), 2))
    graficas = {tuple(secuencia_grados(range(n), c)) for k in range(len(pares) + 1)
                for c in itertools.combinations(pares, k)}
    for s in itertools.combinations_with_replacement(range(n - 1, -1, -1), n):
        assert havel_hakimi(s)[0] == (s in graficas) == nx.is_graphical(s)
        if s in graficas:
            assert secuencia_grados(range(n), realizar(s)) == list(s)
    print(f"{n:>2} {2 ** len(pares):>7} {len(graficas):>20}")

[3, 3, 2, 2, 2, 1, 1]: gráfica
   [3, 3, 2, 2, 2, 1, 1] → [2, 2, 1, 1, 1, 1] → [1, 1, 1, 1, 0] → [1, 1, 0, 0] → [0, 0, 0]
   aristas: [(0, 1), (0, 2), (0, 3), (1, 3), (1, 4), (2, 6), (4, 5)]
[4, 4, 4, 1, 1]: no gráfica
   [4, 4, 4, 1, 1] → [3, 3, 0, 0]
[5, 3, 2, 2, 1, 1]: gráfica
   [5, 3, 2, 2, 1, 1] → [2, 1, 1, 0, 0] → [0, 0, 0, 0]
   aristas: [(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (1, 2), (1, 3)]
[3, 2, 2]: no gráfica
   [3, 2, 2]

 n  grafos  secuencias gráficas
 1       1                    1
 2       2                    2
 3       8                    4
 4      64                   11
 5    1024                   31


## 11. Complemento de un grafo

El complemento $\overline{G}$ tiene los mismos vértices y une justo los pares que $G$ no une. Escribirlo y comprobar que $|E| + |\overline{E}| = \binom{n}{2}$, que $\deg_{\overline{G}}(v) = n - 1 - \deg_G(v)$, que el complemento del complemento es el grafo original y que el de $K_{m,n}$ son dos completos separados, $K_m$ y $K_n$. Un grafo es autocomplementario si es isomorfo a su complemento: comprobar que lo son $P_4$ y $C_5$, y contar por fuerza bruta cuántos hay (sin repetir isomorfos) de hasta 6 vértices.

In [11]:
def complemento(vertices, aristas):
    return normalizar(itertools.combinations(vertices, 2)) - normalizar(aristas)


sin_ferri = complemento(ISLAS, FERRIS)
print(f"Parejas de islas sin ferri directo ({len(sin_ferri)}):", sorted(sin_ferri))
assert len(sin_ferri) + len(FERRIS) == comb(len(ISLAS), 2)
assert sin_ferri == normalizar(nx.complement(nx.Graph(FERRIS)).edges())

m, n = 3, 4
v, a = bipartito_completo(m, n)
assert complemento(v, a) == completo(m)[1] | {(i + m, j + m) for i, j in completo(n)[1]}

rng = random.Random(11)
for _ in range(300):
    k = rng.randint(1, 9)
    v, a = list(range(k)), grafo_aleatorio(k, rng.random(), rng)
    c = complemento(v, a)
    ady, ady_c = a_adyacencia(v, a), a_adyacencia(v, c)
    assert len(a) + len(c) == comb(k, 2)
    assert all(len(ady_c[x]) == k - 1 - len(ady[x]) for x in v)
    assert complemento(v, c) == a

for nombre, (v, a) in [("P(4)", camino(4)), ("C(5)", ciclo(5)), ("C(6)", ciclo(6))]:
    print(f"¿{nombre} es autocomplementario? {isomorfos(v, a, v, complemento(v, a))}")

print(f"\n{'n':>2} {'autocomplementarios':>20}")
for k in range(1, 7):
    v = list(range(k))
    pares = list(itertools.combinations(v, 2))
    representantes = []
    if len(pares) % 2 == 0:
        for c in itertools.combinations(pares, len(pares) // 2):
            a = set(c)
            if isomorfos(v, a, v, complemento(v, a)) and not any(isomorfos(v, a, v, r) for r in representantes):
                representantes.append(a)
    print(f"{k:>2} {len(representantes):>20}")
    assert len(representantes) == [1, 0, 0, 1, 2, 0][k - 1]

Parejas de islas sin ferri directo (12): [('EH', 'FV'), ('EH', 'GC'), ('EH', 'LP'), ('EH', 'LZ'), ('FV', 'LG'), ('FV', 'LP'), ('FV', 'TF'), ('GC', 'LG'), ('GC', 'LP'), ('LG', 'LZ'), ('LP', 'LZ'), ('LZ', 'TF')]
¿P(4) es autocomplementario? True
¿C(5) es autocomplementario? True
¿C(6) es autocomplementario? False

 n  autocomplementarios
 1                    1
 2                    0
 3                    0
 4                    1
 5                    2
 6                    0


## 12. Grafo conexo

Un grafo es conexo si hay un camino entre cualquier par de vértices. Comprobarlo con un BFS desde un vértice cualquiera (como en *Búsqueda en anchura*) y verificar con todos los grafos de hasta 5 vértices dos resultados:

- Si $G$ no es conexo, su complemento sí lo es.
- Todo grafo con $n$ vértices y más de $\binom{n-1}{2}$ aristas es conexo; con $\binom{n-1}{2}$ puede no serlo ($K_{n-1}$ más un vértice aislado).

Contar de paso los grafos conexos con $n$ vértices etiquetados ($1, 1, 4, 38, 728$) y contrastar con `nx.is_connected`.

In [12]:
def es_conexo(vertices, aristas):
    vertices = list(vertices)
    if not vertices:
        return True
    ady = a_adyacencia(vertices, aristas)
    vistos, cola = {vertices[0]}, deque([vertices[0]])
    while cola:
        for v in ady[cola.popleft()] - vistos:
            vistos.add(v)
            cola.append(v)
    return len(vistos) == len(vertices)


print("¿Red de ferris conexa?", es_conexo(ISLAS, FERRIS))
print("¿C(6) conexo?", es_conexo(*ciclo(6)), " ¿Dos triángulos?", es_conexo(*DOS_TRIANGULOS))
print("¿Ferris sin GC-TF conexa?", es_conexo(ISLAS, [a for a in FERRIS if a != ("GC", "TF")]))

print(f"\n{'n':>2} {'grafos':>7} {'conexos':>8}")
for n in range(1, 6):
    v = list(range(n))
    pares = list(itertools.combinations(v, 2))
    conexos = 0
    for k in range(len(pares) + 1):
        for c in itertools.combinations(pares, k):
            conexo = es_conexo(v, c)
            conexos += conexo
            assert conexo == nx.is_connected(a_nx(v, c))
            assert conexo or es_conexo(v, complemento(v, c))
            assert conexo or k <= comb(n - 1, 2)
    print(f"{n:>2} {2 ** len(pares):>7} {conexos:>8}")
    assert conexos == [1, 1, 4, 38, 728][n - 1]
    if n > 1:
        assert not es_conexo(v, completo(n - 1)[1])

¿Red de ferris conexa? True
¿C(6) conexo? True  ¿Dos triángulos? False
¿Ferris sin GC-TF conexa? False

 n  grafos  conexos
 1       1        1
 2       2        1
 3       8        4
 4      64       38
 5    1024      728


## 13. Subgrafo inducido

El subgrafo inducido por un conjunto $S$ de vértices conserva solo las aristas con los dos extremos en $S$. Obtener el de las islas occidentales en la red de ferris, comprobar que cualquier subconjunto de $k$ vértices de $K_n$ induce $K_k$ y que quitar un vértice de $C_n$ deja $P_{n-1}$, y contrastar con `G.subgraph` de `networkx`. Por último, buscar por fuerza bruta los mayores conjuntos de islas que inducen un grafo completo (las cliques máximas).

In [13]:
def subgrafo_inducido(aristas, s):
    return {(u, v) for u, v in normalizar(aristas) if u in s and v in s}


OCCIDENTALES = {"TF", "LG", "LP", "EH"}
print("Ferris entre las occidentales:", sorted(subgrafo_inducido(FERRIS, OCCIDENTALES)))
print("Les falta:", sorted(complemento(sorted(OCCIDENTALES), subgrafo_inducido(FERRIS, OCCIDENTALES))))

rng = random.Random(13)
for _ in range(200):
    n = rng.randint(3, 9)
    s = set(rng.sample(range(n), rng.randint(0, n)))
    assert subgrafo_inducido(completo(n)[1], s) == set(itertools.combinations(sorted(s), 2))
    quitado = rng.randrange(n)
    resto = set(range(n)) - {quitado}
    nuevo = {(quitado + 1 + i) % n: i for i in range(n - 1)}
    ind = subgrafo_inducido(ciclo(n)[1], resto)
    assert normalizar((nuevo[x], nuevo[y]) for x, y in ind) == camino(n - 1)[1]
    a = grafo_aleatorio(n, 0.5, rng)
    assert subgrafo_inducido(a, s) == normalizar(a_nx(range(n), a).subgraph(s).edges())

for k in range(len(ISLAS), 0, -1):
    cliques = [c for c in itertools.combinations(ISLAS, k)
               if len(subgrafo_inducido(FERRIS, set(c))) == comb(k, 2)]
    if cliques:
        print(f"Cliques máximas ({k} islas):", cliques)
        break
assert k == max(len(c) for c in nx.find_cliques(nx.Graph(FERRIS)))

Ferris entre las occidentales: [('EH', 'LG'), ('EH', 'TF'), ('LG', 'LP'), ('LG', 'TF'), ('LP', 'TF')]
Les falta: [('EH', 'LP')]
Cliques máximas (3 islas): [('LZ', 'FV', 'GC'), ('TF', 'LG', 'LP'), ('TF', 'LG', 'EH')]
